# Network Creation and analysis

The following Notebook contains the code used for the creation of the lexical dynamic network and its subsequent analyses. The **dynetx** library is requested.

In [ ]:
# pip install dynetx

In [ ]:
import dynetx as dn
import networkx as nx
import pandas as pd
import pyphen
from dynetx import algorithms as al

## 1. Dynamic graph creation

The code extracts the nodes and the snapshot they belong to from the text embedding created in the ```embedding``` folder. Specifically, the requested structure to create the dynamic network is the following:
```
n1 n2 t1
```
where
- ```n1``` and ```n2``` are nodes
- ```t1``` is the timestamp of interaction appearance


In [44]:
g = dn.DynGraph()

with open("grafo.txt", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        n1, n2, t = line.split()

        g.add_interaction(
            int(n1),
            int(n2),
            int(t)
        )

print("Nodi:", g.number_of_nodes())
print("Interazioni:", g.number_of_interactions())


Nodi: 37
Interazioni: 100


# 2. Node Labelling
For the Complexity labelling, 3 metrics are merged:

1. **Morphological Complexity**: By the Zipf Law (or law of least effort) complex words are inveresely proportional to their length in terms of characters and syllables.
2. **De Mauro's list of common words**: common words can be Fundamentals (high frequency words, 90% coverage of texts), High Usage (6-8% coverage of texts) and High Availability (common but rarely used words). A complex word doesn't belong to any of these sets. 
3. **Inverse Document Frequency**: A generic/less complex word tends to appear more frequently than a complex/high informative word.

In [ ]:
# 1. Setup sillabatore italiano
dic = pyphen.Pyphen(lang='it_IT')

# 2. Caricamento lista De Mauro (Ground Truth)

nvdb = pd.read_csv('nvdb.csv').set_index('lemma')['categoria'].to_dict()

def get_word_complexity(lemma, total_doc_count, doc_freq_of_lemma):
    """
    Restituisce un vettore di complessità per il lemma
    """
    
    # Metrica 1: Lunghezza
    char_len = len(lemma)
    syll_len = len(dic.inserted(lemma).split('-'))
    
    # Metrica 2: Categoria De Mauro (Score arbitrario per ordinamento)
    # 0=Fondamentale (semplice), 1=Alto Uso, 2=Alta Disp, 3=Altro (Complesso)
    cat = nvdb.get(lemma, 'Altro')
    cat_score = {'FO': 0, 'AU': 1, 'AD': 2, 'Altro': 3}.get(cat, 3)
    
    # Metrica 3: Informatività (IDF semplice)
    # Più alto è l'IDF, più la parola è specifica/rara (complessa)
    import math
    idf = math.log(total_doc_count / (1 + doc_freq_of_lemma))
    
    return {
        'char_len': char_len,
        'syll_len': syll_len,
        'demauro_level': cat_score,
        'idf': idf
    }

# Esempio di integrazione nel grafo
# G è il tuo grafo NetworkX/DyNetX
# for node in G.nodes():
#    attrs = get_word_complexity(node, N_DOCS, doc_freqs[node])
#    G.nodes[node].update(attrs)

In [45]:
print(dir(g))
print(dir(al))

['_DynGraph__presence_test', '__class__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__networkx_backend__', '__networkx_cache__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_adj', '_node', 'add_cycle', 'add_edge', 'add_edges_from', 'add_interaction', 'add_interactions_from', 'add_node', 'add_nodes_from', 'add_path', 'add_star', 'add_weighted_edges_from', 'adj', 'adjacency', 'adjlist_inner_dict_factory', 'adjlist_outer_dict_factory', 'avg_number_of_nodes', 'avg_temporal_degree', 'clear', 'clear_edges', 'copy', 'coverage', 'degree', 'degree_iter', 'density', 'directed', 'edge_attr_dict_factory', 'edge_contribution', 'edge_removal', 'edge_subgraph', 'edges', 'edges_iter', 'get_edge_data', 'get

In [ ]:
# qui salviamo il grafo dinamico in un file così da poterlo caricare in futuro senza dover rifare tutto

## 2. Network analysis

In [ ]:
# snapshot ids
times = sorted(g.temporal_snapshots_ids())
times

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [ ]:
# dynamic graph analysis
rows = []

for t in times:
    # snapshot al tempo t (intervallo [t, t])
    Gt = g.time_slice(t, t)

    num_nodes = Gt.number_of_nodes()
    num_edges = Gt.number_of_edges()

    density = nx.density(Gt) if num_nodes > 1 else 0

    avg_degree = (
        sum(dict(Gt.degree()).values()) / num_nodes
        if num_nodes > 0 else 0
    )

    if num_nodes > 0:
        num_components = nx.number_connected_components(Gt)
        largest_cc = max(len(c) for c in nx.connected_components(Gt))
    else:
        num_components = 0
        largest_cc = 0

    clustering = (
        nx.average_clustering(Gt)
        if num_edges > 0 else 0
    )

    rows.append({
        "time": t,
        "nodes": num_nodes,
        "edges": num_edges,
        "density": density,
        "avg_degree": avg_degree,
        "components": num_components,
        "largest_cc": largest_cc,
        "clustering": clustering
    })

pd.options.display.float_format = '{:.4f}'.format
pd.set_option('display.max_width', 1000)
df = pd.DataFrame(rows).sort_values("time")
print(df)


    time  nodes  edges   density  avg_degree  components  largest_cc  \
0      0     12     12  0.181818    2.000000           4           3   
1      1     15      9  0.085714    1.200000           7           3   
2      2     14     11  0.120879    1.571429           3           6   
3      3     18      9  0.058824    1.000000           9           2   
4      4     10     11  0.244444    2.200000           3           4   
5      5      8      8  0.285714    2.000000           1           8   
6      6     18      9  0.058824    1.000000           9           2   
7      7     18      9  0.058824    1.000000           9           2   
8      8      9      9  0.250000    2.000000           3           3   
9      9     12      6  0.090909    1.000000           6           2   
10    10      7      7  0.333333    2.000000           1           7   
11    11     12      6  0.090909    1.000000           6           2   
12    12     11      6  0.109091    1.090909           5        

## 2. Specific analyses

### 2.1 Number of neighbors per node over time (useful for *RQ1*)

In [ ]:
# number of neighbors (degree) per node over time (useful for RQ1)

times = sorted(g.temporal_snapshots_ids())
nodes = sorted(g.nodes())

# grado per ogni nodo
rows = []

for t in times:
    Gt = g.time_slice(t, t)  # snapshot al tempo t

    for v in nodes:
        if v in Gt:
            degree = Gt.degree(v)
        else:
            degree = 0

        rows.append({
            "time": t,
            "node": v,
            "degree": degree
        })



df = pd.DataFrame(rows)

print(df.head(5))

   time  node  degree
0     0     0       2
1     0     1       2
2     0     2       2
3     0     3       2
4     0     4       2


### 2.2 Community detection (useful for *RQ2*)

In [ ]:
from cdlib import algorithms, TemporalClustering

# Supponiamo tu abbia una lista di grafi per ogni snapshot: graphs = [g1960, g1965, ...]

# Metodo: Discovery sui singoli snapshot + Matching Temporale
coms_by_time = []
for g in graphs:
    # Leiden è lo stato dell'arte (meglio di Louvain)
    coms = algorithms.leiden(g) 
    coms_by_time.append(coms)

# Matching temporale per vedere l'evoluzione
# Calcola la similarità tra comunità di t e t+1
matches = TemporalClustering.get_explicit_community_match(coms_by_time, method="jaccard")

# matches ora contiene i link per costruire l'Alluvial Plot